In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from STA_PyTorch import sta_net

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score
import os
import pandas as pd
import openpyxl
from datetime import datetime


# result logging
if not os.path.exists('Results'):
	os.makedirs('Results')
result_file = "Results/Log.txt"
log_excel = 'Results/Log.xlsx'

class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.sampling_points = config['sampling_points']
		self.flooding_level = config['flooding_level']
		self.metric_epochs = config['metric_epochs']

		self.model = sta_net().to(self.device)
		self.criterion = nn.CrossEntropyLoss()
		self.optimizer = torch.optim.AdamW(self.model.parameters())

		os.makedirs('Results', exist_ok=True)
	
	def train_epoch(self, epoch, train_loader):
		self.model.train()
		total_correct, total_loss = 0, 0

		for batch_eeg, batch_nirs, batch_labels in train_loader:
			batch_eeg = batch_eeg.to(self.device, dtype=torch.float32)
			batch_nirs = batch_nirs.to(self.device, dtype=torch.float32)
			batch_labels = batch_labels.to(self.device)
			
			outputs, _ = self.model(batch_eeg, batch_nirs)
			loss = self.criterion(outputs, batch_labels.long())

			if epoch < 30:
				b = self.flooding_level[0]
			elif epoch < 50:
				b = self.flooding_level[1]
			else:
				b = self.flooding_level[2]

			loss = (loss - b).abs() + b
			self. optimizer.zero_grad()
			loss.backward()
			self.optimizer.step()
			
			preds = torch.argmax(outputs, dim=1)
			total_correct += (preds == batch_labels).sum().item()
			total_loss += loss.item()

			train_loss = total_loss / len(train_loader)
			train_acc = total_correct / len(train_loader.dataset)
			# print(f"Training Loss: {train_loss:.2f} | Training Acc: {train_acc:.2f}")
			
			return train_loss, train_acc
	
	def evaluate_epoch(self, epoch, eval_loader):
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_eeg, eval_nirs, eval_labels in eval_loader:
				eval_eeg = eval_eeg.to(self.device, dtype=torch.float32)
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
				eval_labels = eval_labels.to(self.device)
				
				outputs, _ = self.model(eval_eeg, eval_nirs)
				loss = self.criterion(outputs, eval_labels.long())
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				total_loss += loss.item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds, zero_division=0)
		f1 = f1_score(all_labels, all_preds, zero_division=0)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		# print(f"Evaluation Loss: {loss:.2f} | Evaluation Acc: {acc:.2f}")
		return loss, acc, precision, recall, f1, kappa

	def train_subject(self, subject_path, subject):
		seed= 42
		torch.manual_seed(seed)
		
		with np.load(os.path.join(subject_path, subject)) as data:
			eeg = data['eeg']
			nirs = data['fnirs']
			labels = data['label']
		nirs *= 1e3

		print(f'Subject: {subject} EEG {eeg.shape} fNIRS {nirs.shape} Labels {labels.shape}')
		# eeg.shape = [600, 16, 16, 300, 1]
		# nirs.shape = [600, 11, 16, 16, 30, 2]
		# labels.shape = [600,]
	
		eeg = torch.tensor(eeg, dtype=torch.float32)
		nirs = torch.tensor(nirs, dtype=torch.float32)
		labels = torch.tensor(labels, dtype=torch.float32)
		eeg, nirs, labels = eeg.to(self.device), nirs.to(self.device), labels.to(self.device)

		train_eeg, test_eeg, train_labels, test_labels = train_test_split(eeg, labels, test_size=0.4, random_state=self.config['RANDOM_STATE'])
		train_nirs, test_nirs, _, _ = train_test_split(nirs, labels, test_size=0.4, random_state=self.config['RANDOM_STATE'])
		
		train_dataset = TensorDataset(train_eeg, train_nirs, train_labels)
		train_loader = DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		
		eval_dataset = TensorDataset(test_eeg, test_nirs, test_labels)
		eval_loader = DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		
		for epoch in range(self.config['num_epochs']):
			train_loss, train_acc = self.train_epoch(epoch, train_loader)
			eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(epoch, eval_loader)
			
			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject} | Epoch {epoch+1}/{self.config['num_epochs']} | "
				f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | "
				f"Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")

		mean_acc, std_acc = np.mean(acc_list[-self.metric_epochs:]), np.std(acc_list[-self.metric_epochs:])
		mean_precision, std_precision = np.mean(precision_list[-self.metric_epochs:]), np.std(precision_list[-self.metric_epochs:])
		mean_recall, std_recall = np.mean(recall_list[-self.metric_epochs:]), np.std(recall_list[-self.metric_epochs:])
		mean_f1, std_f1 = np.mean(f1_list[-self.metric_epochs:]), np.std(f1_list[-self.metric_epochs:])
		mean_kappa = np.mean(kappa_list[-self.metric_epochs:])

		mean_acc, std_acc = mean_acc * 100, std_acc * 100
		mean_precision, std_precision = mean_precision * 100, std_precision * 100
		mean_recall, std_recall = mean_recall * 100, std_recall * 100
		mean_f1, std_f1 = mean_f1 * 100, std_f1 * 100

		log_excel = 'Results/Log.xlsx'
		new_row = pd.DataFrame([[mean_acc, std_acc, mean_precision, std_precision, mean_recall, std_recall, mean_f1, std_f1, mean_kappa]], 
							columns=['Accuracy', 'Std_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
		new_row = new_row.round(2)
		if not os.path.exists(log_excel):
			new_row.to_excel(log_excel, index=False, engine='openpyxl')
		else:
			with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
				if 'Sheet1' in writer.sheets:
					startrow = writer.sheets['Sheet1'].max_row
				else:
					startrow = 0
				new_row.to_excel(writer, index=False, header=False, startrow=startrow)

		print(f'Subject:{subject} Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')			
		return

# Hyperparameters
config = {
	'depth': 4,
	'query_size': 64,
	'key_size': 64,
	'value_size': 64,
	'emb_size': 64,
	'dict_len': 512,
	'decay': 0.99,
	'num_heads': 4,
	'expansion': 2,
	'conv_dropout': 0.3,
	'self_dropout': 0.3,
	'cross_dropout': 0.3,
	'cls_dropout': 0.5,
	'num_classes': 2,
	'batch_size': 64,
	'num_epochs': 200,
	'metric_epochs': 50,
	'learning_rate': 1e-3,
	'sampling_points': 100,
	'flooding_level': [0.45, 0.40, 0.35],
	'RANDOM_STATE': 38
}

# Initialize and run trainer
trainer = Trainer(config)
subject_path = r'../../Dataset/EF-WG/Model_Input'
subject_list = os.listdir(subject_path)
subject_list.sort(key=lambda x: int(''.join(filter(str.isdigit, x))))
print(subject_list)
for subject in subject_list:
	s = subject.split('.')[0]
	print(f"\n=== Subject {s} ===")
	results = trainer.train_subject(subject_path, subject)

['VP001.npz', 'VP002.npz', 'VP003.npz', 'VP004.npz', 'VP005.npz', 'VP006.npz', 'VP007.npz', 'VP008.npz', 'VP009.npz', 'VP010.npz', 'VP011.npz', 'VP012.npz', 'VP013.npz', 'VP014.npz', 'VP015.npz', 'VP016.npz', 'VP017.npz', 'VP018.npz', 'VP019.npz', 'VP020.npz', 'VP021.npz', 'VP022.npz', 'VP023.npz', 'VP024.npz', 'VP025.npz', 'VP026.npz']

=== Subject VP001 ===
Subject: VP001.npz EEG (600, 16, 16, 600, 1) fNIRS (600, 11, 16, 16, 30, 2) Labels (600, 2)


RuntimeError: Expected 4D (unbatched) or 5D (batched) input to conv3d, but got input of size: [64, 1, 11, 16, 16, 29]